# FraudBench Experiment Runner (Colab)

This notebook runs **GPU-only** (Neural model) experiments on Colab.
Tree model experiments (XGBoost, HopSkipJump, Square) run locally -- see commands in docs/ToDo.md.

**Workflow:**
1. Mount Google Drive for data and results persistence
2. Clone/update the repo
3. Install dependencies
4. Link datasets from Google Drive
5. Run GPU experiments (Neural models x 3 seeds)
6. Save results to Google Drive

In [ ]:
# Cell 1: Verify GPU is available
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    gpu_mem = getattr(props, "total_memory", getattr(props, "total_mem", 0)) / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU")

In [ ]:
# Cell 2: Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = "/content/drive/MyDrive/FraudBench"
for subdir in ["data", "results", "models", "logs"]:
    os.makedirs(os.path.join(DRIVE_ROOT, subdir), exist_ok=True)
    print(f"  {DRIVE_ROOT}/{subdir}/")

print("\nGoogle Drive mounted and directories ready.")

In [ ]:
# Cell 3: Clone or update repo
import os
import shutil

REPO_URL = "https://github.com/iHaydenzZ/Capstone_FraudBench.git"
REPO_DIR = "/content/Capstone_FraudBench"

if os.path.exists(os.path.join(REPO_DIR, ".git")):
    print("Repo exists. Pulling latest changes...")
    os.chdir(REPO_DIR)
    !git pull
else:
    # Always start from /content so git clone has a valid CWD
    os.chdir("/content")
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
        print("Removed stale directory (not a git repo).")
    print("Cloning repo...")
    !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)

print(f"\nWorking directory: {os.getcwd()}")
!git log --oneline -3

In [ ]:
# Cell 4: Install dependencies
# Colab does not have uv, so we use pip install -e . (editable mode)
!pip install -e . -q 2>&1 | tail -5

# Upgrade numba to work with Colab's NumPy 2.x (ART imports brendel_bethge -> numba)
!pip install "numba>=0.61" -q 2>&1 | tail -3

In [ ]:
# Cell 5: Symlink dataset directories from Google Drive
# The loader expects data at <project_root>/datasets/<DatasetName>/
# We symlink individual dirs (not the whole datasets/ folder,
# because it also contains Python source files like loader.py)
import os

DRIVE_DATA = "/content/drive/MyDrive/FraudBench/data"
DATASETS_DIR = "/content/Capstone_FraudBench/datasets"

for dataset_dir in ["CCFD", "ieee-fraud-detection", "LCLD", "Sparkov"]:
    src = os.path.join(DRIVE_DATA, dataset_dir)
    dst = os.path.join(DATASETS_DIR, dataset_dir)
    if os.path.islink(dst):
        os.unlink(dst)
    if os.path.exists(src):
        os.symlink(src, dst)
        files = os.listdir(src)
        print(f"  Linked: {dataset_dir}/ ({len(files)} files)")
    else:
        print(f"  NOT FOUND on Drive: {dataset_dir}/ -- upload to {src}")

print("\nDataset symlinks ready.")

In [ ]:
# Cell 6: Run a single experiment
# Change CONFIG to run different experiments.
# Available configs: ls configs/

CONFIG = "configs/ccfd.yaml"

!python -m runner.run --config {CONFIG}

In [ ]:
# Cell 7: GPU batch -- All Neural model experiments x 3 seeds
# These use PyTorch for model training and CAPGD gradient computation.
# Runtime: T4 GPU (~1.96 units/hr)
# Total: 16 configs x 3 seeds = 48 experiments
import os
import subprocess
import time
from tqdm.notebook import tqdm

SEEDS = [42, 123, 456]

gpu_configs = [
    # --- CCFD (neural) ---
    "configs/ccfd.yaml",                   # neural baseline + CAPGD
    "configs/ccfd_adv_train.yaml",         # neural + adversarial training
    "configs/ccfd_input_val.yaml",         # neural + input validation
    "configs/ccfd_eps_sweep.yaml",         # neural + CAPGD epsilon sweep

    # --- IEEE-CIS (neural) ---
    "configs/ieee_cis_neural.yaml",        # neural baseline + CAPGD
    "configs/ieee_cis_adv_train.yaml",     # neural + adversarial training
    "configs/ieee_cis_input_val.yaml",     # neural + input validation
    "configs/ieee_cis_eps_sweep.yaml",     # neural + CAPGD epsilon sweep

    # --- LCLD (neural) ---
    "configs/lcld.yaml",                   # neural baseline + CAPGD
    "configs/lcld_adv_train.yaml",         # neural + adversarial training
    "configs/lcld_input_val.yaml",         # neural + input validation
    "configs/lcld_eps_sweep.yaml",         # neural + CAPGD epsilon sweep

    # --- Sparkov (neural) ---
    "configs/sparkov.yaml",                # neural baseline + CAPGD
    "configs/sparkov_adv_train.yaml",      # neural + adversarial training
    "configs/sparkov_input_val.yaml",      # neural + input validation
    "configs/sparkov_eps_sweep.yaml",      # neural + CAPGD epsilon sweep
]

def _short_name(path):
    return os.path.splitext(os.path.basename(path))[0]

experiments = [(config, seed) for config in gpu_configs for seed in SEEDS]
failed = []

for config, seed in tqdm(experiments, desc="GPU experiments", unit="exp"):
    start = time.time()
    result = subprocess.run(
        ["python", "-m", "runner.run", "--config", config, "--seed", str(seed)],
        capture_output=True, text=True
    )
    elapsed = time.time() - start

    if result.returncode != 0:
        print(f"\nFAILED ({elapsed:.1f}s): {_short_name(config)} s{seed}")
        print(result.stderr[-500:] if result.stderr else "No error output")
        failed.append(f"{config} --seed {seed}")

total = len(experiments)
print(f"\nGPU batch done: {total - len(failed)}/{total} succeeded.")
if failed:
    print(f"Failed ({len(failed)}):")
    for f in failed:
        print(f"  - {f}")
print("\nRun Cell 8 to save results to Drive.")

## Local Experiments (Tree Models -- Run on Your Machine)

Tree models (XGBoost) don't need GPU. Run these locally with `uv run`:

```bash
# CCFD tree (4 configs x 3 seeds = 12 experiments)
uv run python -m runner.run --config configs/ccfd_tree.yaml --seed 42
uv run python -m runner.run --config configs/ccfd_tree.yaml --seed 123
uv run python -m runner.run --config configs/ccfd_tree.yaml --seed 456
uv run python -m runner.run --config configs/ccfd_tree_input_val.yaml --seed 42
uv run python -m runner.run --config configs/ccfd_tree_input_val.yaml --seed 123
uv run python -m runner.run --config configs/ccfd_tree_input_val.yaml --seed 456
uv run python -m runner.run --config configs/ccfd_tree_hsj.yaml --seed 42
uv run python -m runner.run --config configs/ccfd_tree_hsj.yaml --seed 123
uv run python -m runner.run --config configs/ccfd_tree_hsj.yaml --seed 456
uv run python -m runner.run --config configs/ccfd_tree_square.yaml --seed 42
uv run python -m runner.run --config configs/ccfd_tree_square.yaml --seed 123
uv run python -m runner.run --config configs/ccfd_tree_square.yaml --seed 456

# IEEE-CIS tree (4 configs x 3 seeds = 12 experiments)
uv run python -m runner.run --config configs/ieee_cis.yaml --seed 42
uv run python -m runner.run --config configs/ieee_cis.yaml --seed 123
uv run python -m runner.run --config configs/ieee_cis.yaml --seed 456
uv run python -m runner.run --config configs/ieee_cis_tree_input_val.yaml --seed 42
uv run python -m runner.run --config configs/ieee_cis_tree_input_val.yaml --seed 123
uv run python -m runner.run --config configs/ieee_cis_tree_input_val.yaml --seed 456
uv run python -m runner.run --config configs/ieee_cis_tree_hsj.yaml --seed 42
uv run python -m runner.run --config configs/ieee_cis_tree_hsj.yaml --seed 123
uv run python -m runner.run --config configs/ieee_cis_tree_hsj.yaml --seed 456
uv run python -m runner.run --config configs/ieee_cis_tree_square.yaml --seed 42
uv run python -m runner.run --config configs/ieee_cis_tree_square.yaml --seed 123
uv run python -m runner.run --config configs/ieee_cis_tree_square.yaml --seed 456

# LCLD tree (4 configs x 3 seeds = 12 experiments)
uv run python -m runner.run --config configs/lcld_tree.yaml --seed 42
uv run python -m runner.run --config configs/lcld_tree.yaml --seed 123
uv run python -m runner.run --config configs/lcld_tree.yaml --seed 456
uv run python -m runner.run --config configs/lcld_tree_input_val.yaml --seed 42
uv run python -m runner.run --config configs/lcld_tree_input_val.yaml --seed 123
uv run python -m runner.run --config configs/lcld_tree_input_val.yaml --seed 456
uv run python -m runner.run --config configs/lcld_tree_hsj.yaml --seed 42
uv run python -m runner.run --config configs/lcld_tree_hsj.yaml --seed 123
uv run python -m runner.run --config configs/lcld_tree_hsj.yaml --seed 456
uv run python -m runner.run --config configs/lcld_tree_square.yaml --seed 42
uv run python -m runner.run --config configs/lcld_tree_square.yaml --seed 123
uv run python -m runner.run --config configs/lcld_tree_square.yaml --seed 456

# Sparkov tree (4 configs x 3 seeds = 12 experiments)
uv run python -m runner.run --config configs/sparkov_tree.yaml --seed 42
uv run python -m runner.run --config configs/sparkov_tree.yaml --seed 123
uv run python -m runner.run --config configs/sparkov_tree.yaml --seed 456
uv run python -m runner.run --config configs/sparkov_tree_input_val.yaml --seed 42
uv run python -m runner.run --config configs/sparkov_tree_input_val.yaml --seed 123
uv run python -m runner.run --config configs/sparkov_tree_input_val.yaml --seed 456
uv run python -m runner.run --config configs/sparkov_tree_hsj.yaml --seed 42
uv run python -m runner.run --config configs/sparkov_tree_hsj.yaml --seed 123
uv run python -m runner.run --config configs/sparkov_tree_hsj.yaml --seed 456
uv run python -m runner.run --config configs/sparkov_tree_square.yaml --seed 42
uv run python -m runner.run --config configs/sparkov_tree_square.yaml --seed 123
uv run python -m runner.run --config configs/sparkov_tree_square.yaml --seed 456
```

**Total local: 16 configs x 3 seeds = 48 experiments**

In [ ]:
# Cell 8: Copy results to Google Drive for persistence
import shutil
import glob
import os

RESULTS_SRC = "/content/Capstone_FraudBench/results"
RESULTS_DST = "/content/drive/MyDrive/FraudBench/results"

for f in glob.glob(os.path.join(RESULTS_SRC, "*")):
    if os.path.isfile(f):
        dst = os.path.join(RESULTS_DST, os.path.basename(f))
        shutil.copy2(f, dst)
        print(f"Saved: {os.path.basename(f)}")

# Also copy figures if they exist
figures_src = os.path.join(RESULTS_SRC, "figures")
figures_dst = os.path.join(RESULTS_DST, "figures")
if os.path.isdir(figures_src):
    if os.path.exists(figures_dst):
        shutil.rmtree(figures_dst)
    shutil.copytree(figures_src, figures_dst)
    print("Saved: figures/ directory")

print(f"\nAll results backed up to {RESULTS_DST}")

In [ ]:
# Cell 9: Session cleanup
# IMPORTANT: Run this when done to stop consuming compute units!
# First make sure results are saved (run Cell 8 first!)

print("Make sure you have saved results to Google Drive (Cell 8) before disconnecting!")
print("To disconnect: Runtime > Disconnect and delete runtime")
print("Or uncomment the line below:")
# from google.colab import runtime; runtime.unassign()